# Домашнее задание: Pydantic


## Важно!

- При выполнении задания используем точные типы (`EmailStr`, `HttpUrl`, `SecretStr`, `Decimal`, конкретные `Enum`).
- Придерживаемся принципа разделения валидаций: проверка поля — в `field_validator`, сквозные зависимости — в `model_validator`


## Задача 1. Профиль пользователя (валидация полей)

Постройте модель профиля пользователя для внутренней CRM:

**Требования**
1. Обязательные поля: `id: UUID`, `email: EmailStr`, `name: str`.
2. Опциональные поля: `website: HttpUrl | None`, `bio: str | None`.
3. Пароль хранится как `SecretStr`, должен быть не короче 8 символов.
4. Имя (`name`) нормализуйте: тримминг + одна пробельная последовательность между словами + первая буква каждого слова заглавная.
5. Если указан `website`, домен сайта не должен совпадать с доменом `email` (смысл: личный сайт != корпоративная почта).

Подсказки: используйте `field_validator` для нормализации и локальных проверок; и `model_validator(mode="after")` для проверки зависимости `email` ↔ `website`.


In [18]:
!pip install "pydantic>=2.8,<2.12" "gradio==5.49.1" -q

In [19]:
!pip install "pydantic[email,timezone]>=2.8,<2.12" -q

In [20]:
from typing import Optional
from pydantic import BaseModel, Field, EmailStr, HttpUrl, SecretStr
from pydantic import field_validator, model_validator
from uuid import UUID



class UserProfile(BaseModel):
    # TODO: опишите поля согласно требованиям
    id: UUID
    email: EmailStr
    name: str
    website: Optional[HttpUrl] = None
    bio: Optional[str] = None
    password: SecretStr

    # TODO: нормализация имени
    @field_validator("name")
    @classmethod
    def normalize_name(cls, v: str) -> str:
        return ' '.join(v.strip().split()).title()

    # TODO: проверка длины пароля
    @field_validator("password")
    @classmethod
    def password_strength(cls, v: SecretStr) -> SecretStr:
        if len(v.get_secret_value()) < 8:
            raise ValueError("Пароль должен быть не короче 8 символов")
        return v

    # TODO: сквозная проверка доменов email/website
    @model_validator(mode="after")
    def check_domains(self):
        if self.website is None:
            return  # Если сайта нет — пропускаем проверку
        email_domain = self.email.split('@')[1]
        website_domain = urlparse(self.website).netloc
        if email_domain == website_domain:
            raise ValueError("Домен сайта не должен совпадать с доменом email")
        return self


## Задача 2. Валидация функции заказа (`@validate_call`)

Реализуйте функцию `place_order`, которая принимает:
- `user_id: UUID`
- `sku: str` (артикул, только заглавные буквы/цифры, длина 3–12)
- `quantity: int` (>0)
- `price: Decimal` (>= 0), округляется банковским методом до 2 знаков

Функция должна возвращать словарь с ключами: `user_id`, `sku`, `quantity`, `price`, `amount` (quantity × price).

Используйте `@validate_call` и локальные проверки через обычный код (или вспомогательные валидаторы `TypeAdapter` не используем).


In [21]:
from pydantic import validate_call
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
import re

# TODO: реализуйте функцию с @validate_call
@validate_call
def place_order(user_id, sku, quantity, price):
   # sku: только заглавные буквы и цифры, длина 3–12
    if not re.fullmatch(r'[A-Z0-9]{3,12}', sku):
        raise ValueError('sku должен содержать только заглавные буквы и цифры, длина 3–12 символов')

    # quantity > 0
    if quantity <= 0:
        raise ValueError('quantity должен быть больше 0')

    # price >= 0, округление банковским методом до 2 знаков
    if price < 0:
        raise ValueError('price не может быть отрицательным')

    price = price.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    amount = (price * quantity).quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)

    return {
        'user_id': user_id,
        'sku': sku,
        'quantity': quantity,
        'price': price,
        'amount': amount,
    }


## Задача 3. Модель заказа с бизнес-правилами

Смоделируйте заказ в магазине цифровых товаров.

**Требования**
- `OrderStatus: Enum` со значениями `new`, `paid`, `delivered`, `canceled`.
- Модель `OrderItem`:
  - `sku: str` как в задаче 2
  - `qty: int` (>0)
  - `unit_price: Decimal` (>=0) округление до 2 знаков
- Модель `Order`:
  - `id: UUID`
  - `user_email: EmailStr`
  - `items: list[OrderItem]` (не пустой)
  - `status: OrderStatus = 'new'`
  - `created_at: datetime` (по умолчанию `datetime.utcnow`)
  - Расчитанное поле `total: Decimal` — сумма по всем позициям
  - В `model_validator(mode="after")` запретите переход в `paid`/`delivered` при `total == 0` и запретите пустые корзины.

**Важно:** используйте только инструменты `pydantic` и стандартную библиотеку.


In [22]:
from pydantic import BaseModel, EmailStr, field_validator, model_validator, Field
from typing import List
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
from datetime import datetime
from enum import Enum
import re

SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")

class OrderStatus(str, Enum):
    # TODO: перечислите статусы
    NEW = "new"
    PAID = "paid"
    DELIVERED = "delivered"
    CANCELED = "canceled"

class OrderItem(BaseModel):
    # TODO: опишите поля
    sku: str
    qty: int
    unit_price: Decimal

    @field_validator("sku")
    @classmethod
    def sku_format(cls, v: str) -> str:
        if not SKU_RE.fullmatch(v):
            raise ValueError("sku должен содержать только A-Z и 0-9, длина 3–12")
        return v

    @field_validator("qty")
    @classmethod
    def qty_positive(cls, v: int) -> int:
        def qty_positive(cls, v: int) -> int:
            if v <= 0:
                raise ValueError("qty должен быть > 0")
        return v

    @field_validator("unit_price")
    @classmethod
    def price_non_negative(cls, v: Decimal) -> Decimal:
        if v < 0:
            raise ValueError("unit_price не может быть отрицательным")
        # округление банковским методом до 2 знаков
        return v.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

class Order(BaseModel):
    # TODO: опишите поля
    id: UUID
    user_email: EmailStr
    items: List[OrderItem]
    status: OrderStatus = OrderStatus.NEW
    created_at: datetime = Field(default_factory=datetime.utcnow)
    total: Decimal = Decimal("0.00")

    @model_validator(mode="after")
    def check_business_rules(self):
        # запрет пустой корзины
        if not self.items:
            raise ValueError("Заказ не может быть без позиций (items пустой).")

        # пересчёт total как сумма qty * unit_price
        total = Decimal("0.00")
        for item in self.items:
            total += item.unit_price * item.qty

        total = total.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)
        self.total = total

        # запрет статусов paid / delivered при total == 0
        if self.total == 0 and self.status in {OrderStatus.PAID, OrderStatus.DELIVERED}:
            raise ValueError("Нельзя перевести заказ с нулевой суммой в статус paid/delivered.")
        return self


## Задача 4. Конфигурация приложения (`BaseSettings`)

Опишите настройки подключения к внешнему API:

- `APISettings(BaseSettings)` с полями:
  - `base_url: HttpUrl`
  - `token: SecretStr`
  - `timeout_sec: int = 5` (1–60)
  - `retries: int = 2` (0–10)
- Используйте `model_config = ConfigDict(env_prefix="API_", env_file=".env", extra="ignore")`
- Проверьте, что значения корректно читаются из переменных окружения.

В тесте ниже среда заполняется вручную.


In [23]:
import os
from pydantic_settings import BaseSettings
from pydantic import ConfigDict, SecretStr, HttpUrl, field_validator

class APISettings(BaseSettings):
    # TODO: поля и валидации
    base_url: HttpUrl
    token: SecretStr
    timeout_sec: int = 5   # 1–60
    retries: int = 2       # 0–10

    # пример проверки диапазона для timeout_sec / retries
    @field_validator("timeout_sec", "retries")
    @classmethod
    def check_ranges(cls, v: int, info):
        if info.field_name == "timeout_sec":
            if not (1 <= v <= 60):
                raise ValueError("timeout_sec должен быть в диапазоне 1–60")
        elif info.field_name == "retries":
            if not (0 <= v <= 10):
                raise ValueError("retries должен быть в диапазоне 0–10")
        return v

    model_config = ConfigDict(
        # TODO: настройте env_prefix и прочие опции
        env_prefix="API_",
        env_file=".env",
        extra="ignore",
    )


## Задача 5. Извлечение из ORM (`from_attributes=True`)

Создайте простую SQLAlchemy-модель `SAUser(id, email, is_active)` (in-memory, без БД) и соответствующую модель Pydantic:

- Pydantic-модель `UserOut` с полями `id: UUID`, `email: EmailStr`, `is_active: bool`.
- Включите поддержку `from_attributes` в `model_config`.
- Создайте инстанс `SAUser` и провалидируйте его через `UserOut.model_validate(sa_user_instance)`.

Проверьте, что преобразование сработало.


In [24]:
from typing import Optional
from sqlalchemy import Column, String, Boolean
from sqlalchemy.orm import declarative_base
from uuid import uuid4
from pydantic import BaseModel, EmailStr, ConfigDict, ValidationError

Base = declarative_base()

class SAUser(Base):
    __tablename__ = "users"
    id = Column(String, primary_key=True, default=lambda: str(uuid4()))
    email = Column(String, nullable=False)
    is_active = Column(Boolean, default=True)

    def __init__(self, email: str, is_active: bool = True):
        self.id = str(uuid4())
        self.email = email
        self.is_active = is_active

class UserOut(BaseModel):
    # TODO: опишите поля и включите from_attributes
    id: str         # или UUID, если хотите сразу приводить тип
    email: EmailStr
    is_active: bool
    model_config = ConfigDict(
        # TODO: включите режим атрибутов
        from_attributes=True
    )


## Задача 6. JSON Schema и дружелюбные ошибки

1. Для модели из задачи 3 сгенерируйте JSON Schema (метод `model_json_schema`) и запишите его в переменную `ORDER_SCHEMA`.
2. Реализуйте функцию `safe_create_order(data: dict) -> tuple[bool, str]`, которая:
   - пытается создать `Order` из входного `dict`,
   - при успехе возвращает `(True, "<total=...>")`,
   - при ошибке возвращает `(False, "<короткое сообщение об ошибке>")` без стек-трейса.

Не используйте сторонние библиотеки.


In [25]:
# Используем модели из задачи 3: OrderStatus, OrderItem, Order

ORDER_SCHEMA = Order.model_json_schema()  # TODO: сгенерируйте схему

def safe_create_order(data: dict) -> tuple[bool, str]:
    # TODO: реализуйте безопасное создание заказа
    try:
        order = Order.model_validate(data)
        # успех
        return True, f"total={order.total}"
    except ValidationError as e:
        # берём короткое, «дружелюбное» сообщение
        # можно взять первое сообщение об ошибке
        first_err = e.errors()[0] if e.errors() else None

        if first_err:
            loc = ".".join(str(x) for x in first_err.get("loc", ()))
            msg = first_err.get("msg", "Validation error")
            if loc:
                error_text = f"{loc}: {msg}"
            else:
                error_text = msg
        else:
            error_text = "Validation error"

        return False, error_text
    except Exception as e:
        # на всякий случай, без стек-трейса
        return False, str(e)
